# __Replication of the DyLoPro case studies on the Road Traffic Fine Management log__

__Paper__: B. Wuyts, H. Weytjens, S. vanden Broucke, J. De Weerdt, *DyLoPro: Profiling the Dynamics of Event Logs* (BPM 2023). Preprint in the [DyLoPro repository](https://github.com/BrechtWts/DyLoPro).

__Experiment replicated__: the case-study evaluation in Sect. 5 of the paper, documented in full in the authors' annotated notebooks [`BPIC_17.ipynb` and `BPIC_19.ipynb`](https://github.com/BrechtWts/DyLoPro_CaseStudies). We apply the same analysis to a third public log, the __Road Traffic Fine Management (RTFM)__ log. The authors mentioned an RTFM case study in that repository, however never published it.

__Claim tested__: applying DyLoPro to a large real-life event log *"demonstrates the effectiveness of the framework"* in discovering patterns, trends and drifts from several process perspectives before other process mining techniques are applied (abstract; Sect. 5).

__The log__: an Italian local police information system for road traffic fines. __150,370 cases__ (one per fine), __561,470 events__ and __11 activities__, for fines created between __January 2000 and June 2013__. Download it [here](https://doi.org/10.4121/uuid:270fd440-1057-4fb9-89a9-b699b47990f5).

The process in brief: a fine is created (*Create Fine*). If it is not paid on the spot, it is sent to the offender (*Send Fine*) and the notification is registered (*Insert Fine Notification*). Unpaid fines get a penalty (*Add penalty*) and can end up with a collection agency (*Send for Credit Collection*). Offenders can pay (*Payment*, possibly in several instalments) or appeal. 

__Structure__ (combining the BPIC17 and BPIC19 notebooks; each section mimics the original section it mirrors):
- __Part I — Setup and reproducibility__: environment, configuration, data checksum, preprocessing (ours and the authors'), DyLoPro initialisation.
- __Part II — Dynamic profiling__: 0 general observations · 1 variants · 2 directly-follows relations · 3 categorical case features · 4 numerical case features · 5 categorical event features · 6 numerical event features · 7 one-year weekly zoom · 8 robustness check.
- __Part III — Replication tables__: coverage, setup comparison, log statistics, findings comparison, deviation log, export manifest.

### How to run
1. Put this notebook and `Road_Traffic_Fine_Management_Process.xes.gz` in the same folder.
2. Create an environment: `conda create -n dlp_rtfm python=3.11 -y`, then `conda activate dlp_rtfm`, then `pip install DyLoPro pm4py "pandas>=2.0,<3.0" notebook`.
3. Start `jupyter notebook` in that folder, open this notebook, fill in `DOWNLOAD_DATE` in cell I.2 and choose *Run All*.

It saves every figure to `figures/` (PNG) and every table to `results/` (CSV), plus `results/run_manifest.json` with versions, checksum and parameters.

# __Part I — Setup and reproducibility__
*Report sections 3 (Replication approach) and 4 (Steps taken).*

### I.1 Environment and library versions

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, platform, json, hashlib, datetime as dt
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pm4py
import DyLoPro as dlp
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 200)

ENV = {
    'run date': dt.datetime.now().strftime('%Y-%m-%d %H:%M'),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'processor': platform.processor() or platform.machine(),
    'DyLoPro': version('DyLoPro'),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'matplotlib': matplotlib.__version__,
    'pm4py': version('pm4py'),
}
assert int(pd.__version__.split('.')[0]) < 3, 'DyLoPro 0.1.1 does not work with pandas 3: pip install "pandas>=2.0,<3.0"'
pd.DataFrame.from_dict(ENV, orient='index', columns=['value'])

### I.2 Configuration
All settings used anywhere in this notebook are defined here. Fill in `DOWNLOAD_DATE`. 

In [ ]:
# data
DATA_PATH = Path('Road_Traffic_Fine_Management_Process.xes.gz')
DATA_DOI = 'https://doi.org/10.4121/uuid:270fd440-1057-4fb9-89a9-b699b47990f5'
DOWNLOAD_DATE = '2026-10-26'          

# original material we replicate
PAPER = 'Wuyts, Weytjens, vanden Broucke, De Weerdt: DyLoPro: Profiling the Dynamics of Event Logs (BPM 2023), preprint in the DyLoPro repository'
ORIGINAL_CODE = {
    'DyLoPro library (github.com/BrechtWts/DyLoPro)': '2d88d48db881af192c78b00fc12afec919a026d1 (2024-07-23); PyPI release 0.1.1',
    'Case-study notebooks (github.com/BrechtWts/DyLoPro_CaseStudies)': '1b5fe667aab0aafae05a1fe86fc3ce78f0ea3b6b (2023-07-28)',
}

# DyLoPro parameters (paper Sect. 4.1-4.2) 
FREQ = 'monthly'                # interval length p for the full log (original: 'weekly', see deviation log)
TIME_UNIT = 'days'
CASE_ASSIGNMENT = 'first_event'
NUMERIC_AGG = 'mean'
MAX_K = 10

# extra analyses 
ZOOM_YEAR = 2009                
ZOOM_FREQ = 'weekly'
ROBUSTNESS_CUTOFF = '2011-01-01' 

# output 
SAVE_FIGURES = True
FIG_DIR, RES_DIR = Path('figures'), Path('results')
FIG_DPI_INLINE, FIG_DPI_EXPORT = 100, 120

plt.rcParams['figure.dpi'] = FIG_DPI_INLINE
common = dict(frequency=FREQ, time_unit=TIME_UNIT, case_assignment=CASE_ASSIGNMENT, numeric_agg=NUMERIC_AGG)
FIG_DIR.mkdir(exist_ok=True); RES_DIR.mkdir(exist_ok=True)
print('DyLoPro has no random components, so no seeds are needed: every run on the same data gives the same figures.')

### I.3 Export helpers
DyLoPro draws every figure with `plt.show()`. The helper below saves each figure to `figures/` just before it is shown, under the name passed to `fig(...)`. `save_table(...)` writes a table to `results/` as CSV and displays it.

In [ ]:
_next_fig = {'name': None}
_original_show = plt.show

def fig(name):
    """Name the next DyLoPro figure; it is saved as figures/<name>.png when shown."""
    _next_fig['name'] = name

def _show_and_save(*args, **kwargs):
    if SAVE_FIGURES and _next_fig['name']:
        for i, num in enumerate(plt.get_fignums()):
            suffix = '' if i == 0 else f'_{i}'
            plt.figure(num).savefig(FIG_DIR / f"{_next_fig['name']}{suffix}.png", dpi=FIG_DPI_EXPORT, bbox_inches='tight')
        _next_fig['name'] = None
    return _original_show(*args, **kwargs)

plt.show = _show_and_save

def save_table(df, name, index=True):
    df.to_csv(RES_DIR / f'{name}.csv', index=index)
    return df

### I.4 Loading the event log (step 1: download, step 2: load)

In [ ]:
assert DATA_PATH.exists(), f'{DATA_PATH} not found: put the .xes.gz file in the same folder as this notebook'
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
raw_log = pm4py.read_xes(str(DATA_PATH))
if not isinstance(raw_log, pd.DataFrame):
    raw_log = pm4py.convert_to_dataframe(raw_log)
print(f'file: {DATA_PATH} ({DATA_PATH.stat().st_size:,} bytes), SHA-256 {DATA_SHA256}')
print(f"{raw_log['case:concept:name'].nunique():,} cases, {len(raw_log):,} events, {raw_log['concept:name'].nunique()} activities")
raw_log.head(8)

### I.5 Preprocessing
The RTFM log needs four preprocessing steps before DyLoPro can use it. The authors' own `preprocess_RTFM()` (I.5.a) only does steps 2 and 4 partially; the differences are listed in the deviation log (III.5).

1. __Timestamps.__ Events carry a calendar date only, stored as local midnight (22:00 or 23:00 UTC). Without correction, a fine created on the 1st of a month is assigned to the previous month. We convert each timestamp back to its Italian calendar date.
1. __Case features.__ The XES file stores the fine's attributes (`amount`, `points`, `vehicleClass`, `article`) on the *Create Fine* event only. We copy them to every event of the case as `case:` columns, so DyLoPro can treat them as case features.
1. __Outcome.__ As in BPIC17, we use a 3-class outcome and split it into three binary columns (one-vs-rest), as in [Teinemaa et al.](https://doi.org/10.1145/3301300):
    - `'Credit Collection'` → `'case credit collection'`: the fine was sent for credit collection at some point (39.2% of cases).
    - `'Paid'` → `'case paid'`: at least one payment was registered and the fine never went to credit collection (44.9%). About 90% of these fines were paid in full.
    - `'Unresolved'` → `'case unresolved'`: neither of the above (15.8%). Mostly fines whose last event is *Send Fine*, plus fines that ended through an appeal.

    The categorical case feature `'case:outcome'` holds the three levels. Unlike BPIC17, no cases have to be discarded: every fine falls into exactly one class.
1. __Data types.__ Categorical features are cast to the `object` dtype, which DyLoPro requires.

#### I.5.a The authors' own (unpublished) RTFM preprocessing
`preprocess_case_studies.py` in the case-study repository (commit `1b5fe66`) already contains a `preprocess_RTFM()` function, although no RTFM notebook was ever published. It is reproduced verbatim below for reference; we only use it to quantify how our preprocessing differs.

In [ ]:
def preprocess_RTFM_authors(log):
    to_object = {'org:resource': object, 'article': object, 'points': object}
    log = log.astype(to_object)
    log['time:timestamp'] = pd.to_datetime(log['time:timestamp'], utc=True)
    log = log.drop('matricola', axis=1)
    return log

#### I.5.b Our preprocessing (step 3)

In [ ]:
CASE_ID, ACT, TS = 'case:concept:name', 'concept:name', 'time:timestamp'

def preprocess_rtfm(log: pd.DataFrame) -> pd.DataFrame:
    log = log.copy()

    # Convert back to the Italian calendar date so cases land in the right week/month.
    ts = pd.to_datetime(log[TS], utc=True).dt.tz_convert('Europe/Rome').dt.tz_localize(None).dt.normalize()
    log[TS] = ts.dt.tz_localize('UTC')
    log[CASE_ID] = log[CASE_ID].astype(str)
    log = log.sort_values([CASE_ID, TS], kind='mergesort').reset_index(drop=True)

    # Copy case attributes to every event of the case.
    create = log[log[ACT] == 'Create Fine'].drop_duplicates(CASE_ID).set_index(CASE_ID)
    log['case:amount'] = log[CASE_ID].map(create['amount'])
    log['case:points'] = log[CASE_ID].map(create['points'])
    log['case:vehicleClass'] = log[CASE_ID].map(create['vehicleClass']).astype(object)
    log['case:article'] = log[CASE_ID].map(create['article']).map(lambda x: f'Art. {int(x)}' if pd.notna(x) else np.nan).astype(object)

    # Outcome
    acts = log.groupby(CASE_ID)[ACT].agg(set)
    cc = acts.map(lambda s: 'Send for Credit Collection' in s)
    paid = acts.map(lambda s: 'Payment' in s)
    outcome = pd.Series(np.select([cc, paid], ['Credit Collection', 'Paid'], 'Unresolved'), index=acts.index)
    log['case:outcome'] = log[CASE_ID].map(outcome).astype(object)
    log['case paid'] = (log['case:outcome'] == 'Paid').astype(int)
    log['case credit collection'] = (log['case:outcome'] == 'Credit Collection').astype(int)
    log['case unresolved'] = (log['case:outcome'] == 'Unresolved').astype(int)

    # Categorical event features as object dtype
    for c in ['org:resource', 'dismissal', 'lastSent', 'notificationType', 'vehicleClass']:
        log[c] = log[c].astype(object)
    return log

log = preprocess_rtfm(raw_log)
log.drop_duplicates(CASE_ID)['case:outcome'].value_counts().to_frame().assign(share=lambda d: (d['count'] / d['count'].sum()).round(3))

#### I.5.c Impact of the timestamp correction (quantified for the deviation log)

In [ ]:
authors_log = preprocess_RTFM_authors(raw_log)
first_authors = authors_log.groupby(CASE_ID)[TS].min()
first_ours = log.groupby(CASE_ID)[TS].min().reindex(first_authors.index)
naive = lambda s: s.dt.tz_convert('UTC').dt.tz_localize(None)
MONTH_SHIFT = (naive(first_authors).dt.to_period('M') != naive(first_ours).dt.to_period('M')).mean()
WEEK_SHIFT = (naive(first_authors).dt.to_period('W') != naive(first_ours).dt.to_period('W')).mean()
print(f'With the authors\' UTC timestamps, {MONTH_SHIFT:.1%} of fines fall in a different month and {WEEK_SHIFT:.1%} in a different week than their actual creation date.')
del authors_log

### I.6 Specifying the available features

| Type | Features | Note |
|---|---|---|
| Categorical case | `case:vehicleClass`, `case:article`, `case:outcome` | `case:article` is the article of the Italian Highway Code that was violated (66 levels) |
| Numerical case | `case:amount`, `case:points` | Fine amount at creation (EUR); licence penalty points |
| Categorical event | `org:resource`, `concept:name`, `lastSent`, `dismissal` | `org:resource` is only recorded on *Create Fine* (and *Appeal to Judge*), so it is effectively the officer who issued the fine |
| Numerical event | `expense`, `amount`, `paymentAmount`, `totalPaymentAmount` | `amount` appears on *Create Fine* and *Add penalty*; see Section 6 |

`totalPaymentAmount` is a running total of the payments; we use it with the `'last'` abstraction (Section 6.3), mirroring BPIC19's cumulative feature. Left out: `notificationType` (99.8% one value) and `matricola` (always 0, also dropped by the authors).

In [ ]:
categorical_casefeatures = ['case:vehicleClass', 'case:article', 'case:outcome']
numeric_casefeatures = ['case:amount', 'case:points']
categorical_eventfeatures = ['org:resource', 'concept:name', 'lastSent', 'dismissal']
numeric_eventfeatures = ['expense', 'amount', 'paymentAmount', 'totalPaymentAmount']

### I.7 Initialising DyLoPro (step 4)
As DyLoPro accepts one binary outcome at a time, we start with `'case paid'` and switch with `set_outcomeColumn(outcome)` throughout, exactly as the BPIC17 notebook does.

In [ ]:
rtfm_plots = dlp.DynamicLogPlots(log, case_id_key=CASE_ID, activity_key=ACT, timestamp_key=TS,
                                  categorical_casefeatures=categorical_casefeatures,
                                  numerical_casefeatures=numeric_casefeatures,
                                  categorical_eventfeatures=categorical_eventfeatures,
                                  numerical_eventfeatures=numeric_eventfeatures, outcome='case paid')

All of __DyLoPro__'s plotting methods build time series by computing measures for a chronologically ordered set of sublogs. Each case is assigned to the interval in which its first event (*Create Fine*) occurs.

__Main difference with the originals: the frequency.__ BPIC17 and BPIC19 each cover one year and use `frequency='weekly'`. RTFM covers 13.5 years, so weekly intervals would give about 700 points per series. We use __`frequency='monthly'`__ (162 intervals of roughly 900 fines each) unless specified otherwise. For rare levels we switch to `'quarterly'` to reduce noise. Section 7 repeats key plots for one year (2009) at the originals' weekly resolution. This follows paper Sect. 4.1, footnote 3: each sublog should be populated enough to give representative measures, but not so large that it levels out patterns.

__NOTE__: For the RTFM plots, we use the following settings unless specified otherwise:
- time_unit = 'days'
- frequency = 'monthly'
- case_assignment = 'first_event'
- numeric_agg = 'mean'
- max_k = 10
- xtr_outlier_rem = True

# __Part II — Dynamic profiling of the RTFM log__
*Report section 5 (Results). Every section header names the matching section of the original notebooks.*

## __0. General observations and remarks__ 
*[BPIC17 §0 | BPIC19 §0]*

1. The top panel of every plot again gives a global overview: the number of fines created per month, plus the overall mean TT, mean NEPC or fraction of positive cases.

1. __No lifecycle duplication.__ All events have `lifecycle:transition = 'complete'`, so each activity execution is one event. The BPIC17 caveats about inflated variants and NEPC do not apply here. Case length is short: 2 to 20 events, median 5.

1. __Strong seasonality.__ About 36% of all fines are created in July and August, against roughly 4–5% in each winter month (Section 0.3). The monthly case count therefore swings between a few hundred and over 2,500 every year. Summer months give large, representative samples; winter months give smaller, noisier ones.

1. __Credit collection happens in batches.__ Fines are not sent for credit collection one by one. All 59,013 transfers fall in just 11 months (2002-04, 2003-01, 2004-01, 2004-12, 2006-02, 2007-02, 2009-03, 2010-10, 2012-03, 2013-04, plus 247 fines in 2001-04). No fines were transferred in 2005, 2008 or 2011. A fine reaches credit collection at the earliest 273 days after creation, with a median of 638 days (Section 0.2). This batch processing leaves a clear signature in the TT plots: a __saw-tooth pattern__. All unpaid fines that are old enough are transferred in the same batch, so their TT is simply the time from creation to that batch date. Fines created right after the previous cut-off wait the longest, and each later month waits a little less. The mean TT of credit-collection cases therefore falls in a straight line for consecutive creation months, then jumps up where the fines start belonging to the next batch (see Section 3.1.1).

1. __Bias at the end of the log, this time in the outcome.__ The log was extracted around June 2013. Because credit collection needs at least 273 days and happens in batches, fines created from early 2012 onwards had no chance to reach credit collection yet. The share of credit-collection cases drops from 44% (2011) to 4% (2012) and 0% (2013) (Section 0.1). At the same time, mean TT collapses from roughly 350 days to 30–70 days, and mean NEPC drops from about 3.9 to 2–3 events. In BPIC19 (paper Sect. 5) and BPIC17 the end bias mainly distorted throughput time. In RTFM it distorts __everything__ that depends on how a fine ends: outcome shares, TT, NEPC, variant shares and even activity frequencies.
    - __Rule we follow__: we do not report changes in 2012–2013 as real dynamics. Section 8 repeats key plots for fines created in 2000–2010 only, and there the collapse disappears.
    - The 'Unresolved' class is partly a symptom of the same bias: some of these fines are simply still running.

1. __Bias at the start of the log__ is small. *Create Fine* is always the first event, so every case is complete from its creation onwards. The first months only show slightly fewer fines.

1. __General trends__ (fines created 2000–2011):
    - __Throughput time__: noisy around 300–450 days, with no clear long-term trend. Its swings are driven mainly by the batch timing of credit collection.
    - __Case length (NEPC)__: falls from about 4.4 events (2000) to about 3.5 (2005–2009). This mirrors the falling share of credit-collection cases (5 events each) over the same period.
    - __Outcome__: the credit-collection share falls from about 60% (2000) to about 30% (2005–2006), then rises again to about 44% (2011). The 'Unresolved' share jumps from about 5% to about 20% during 2001–2002 and stays there.

#### 0.1 Supporting analysis: outcome, throughput time and case length per year of fine creation

In [ ]:
cases = log.groupby(CASE_ID).agg(start=(TS, 'min'), end=(TS, 'max'), n_events=(ACT, 'size'),
                                 outcome=('case:outcome', 'first'))
cases['TT_days'] = (cases['end'] - cases['start']).dt.days
cases['year'] = cases['start'].dt.year
per_year = pd.crosstab(cases['year'], cases['outcome'], normalize='index').round(3)
per_year.insert(0, 'cases', cases.groupby('year').size())
per_year['mean TT (days)'] = cases.groupby('year')['TT_days'].mean().round(0)
per_year['mean NEPC'] = cases.groupby('year')['n_events'].mean().round(2)
save_table(per_year, 'table_0_1_per_year')

The helper below summarises, for every level of a feature, how many cases have it and their mean TT, NEPC and outcome shares over the whole log. It backs up the observations under each plot. For event features, a case "has" a level if at least one of its events has it, exactly as in DyLoPro (paper Table 1).

In [ ]:
def level_summary(feature, k=MAX_K):
    has = log.loc[log[feature].notna(), [CASE_ID, feature]].drop_duplicates()
    m = has.merge(cases, left_on=CASE_ID, right_index=True)
    out = m.groupby(feature).agg(cases=(CASE_ID, 'size'), mean_TT_days=('TT_days', 'mean'), mean_NEPC=('n_events', 'mean'))
    out['share_of_cases'] = out['cases'] / len(cases)
    out = out.join(pd.crosstab(m[feature], m['outcome'], normalize='index'))
    out = out.sort_values('cases', ascending=False).head(k).round(3)
    return save_table(out, 'levels_' + feature.replace(':', '_'))

#### 0.2 Supporting analysis: when are fines sent for credit collection?

In [ ]:
cc_events = log[log[ACT] == 'Send for Credit Collection']
cc_per_month = cc_events.groupby(cc_events[TS].dt.to_period('M')).size()
print('Months in which at least one fine was sent for credit collection:', len(cc_per_month))
print(cc_per_month.sort_values(ascending=False).to_string())
days_to_cc = (cc_events.groupby(CASE_ID)[TS].min() - cases['start']).dt.days.dropna()
print('\nDays from fine creation to credit collection:'); print(days_to_cc.describe().round(0).to_string())

#### 0.3 Supporting analysis: seasonality of fine creation

In [ ]:
by_month = cases.groupby(cases['start'].dt.month).agg(share_of_fines=('outcome', 'size'))
by_month['share_of_fines'] = (by_month['share_of_fines'] / len(cases)).round(3)
by_month.index.name = 'month of year'
save_table(by_month.T, 'table_0_3_seasonality')

#### 0.4 Global overview: distinct variants over time
`distinct_variants_AdvancedEvol` replaces `distinct_variants_evol`, which the library deprecates from version 0.1.1 (HISTORY.md). Neither original notebook uses this plot.

In [ ]:
fig('00_4_distinct_variants_tt')
rtfm_plots.distinct_variants_AdvancedEvol(plt_type='type_tt', **common)

__Observations (0.4)__:
- Each month contains about 10–40 distinct variants, and the number moves with the case volume: summer months, with up to 2,500 fines, show the most variants.
- New variants, never seen before, appear in most months but at most about 8 per month, and they cover well under 1% of each month's fines (bottom left). After the first months the process has no real "variant churn": almost every fine follows a path already seen earlier.
- Fines in new variants usually take much longer (often 500–1,500 days, bottom right). They are rare paths, such as appeals or multiple payments.
- This contrasts with BPIC17, where more than 15,000 variants for 31,413 cases means almost every case is new (BPIC17 nb §1).

## __1. Variants__ 
*[BPIC17 §1 | BPIC19 §1]*
In sharp contrast to BPIC17 (over 15,000 variants for about 31,000 cases), RTFM is __highly structured__. It has only __231 variants__ for 150,370 cases, and the top 10 variants cover __98.1%__ of all cases. The three most frequent variants alone cover 82%:
1. *Create Fine → Send Fine → Insert Fine Notification → Add penalty → Send for Credit Collection* (37.6%)
2. *Create Fine → Payment*, paid without the fine being sent (30.8%)
3. *Create Fine → Send Fine*, ending right after sending (13.6%)

Variant-level analysis is therefore much more informative here than in BPIC17. Interesting observations:

1. __Throughput time__:
    - Variant 2 (immediate payment) has a mean TT of only a few days, against several hundred days for all other cases. Variant 3 ends at *Send Fine*, about 90 days after creation.
    - Variants 1, 9 and 10 (all ending in credit collection) show the __saw-tooth pattern__ caused by the batches discussed in Section 0.
1. __Sudden control-flow drifts__ (left-hand panels):
    - __Variants 6 and 10 disappear at the end of 2006.__ Both contain *Payment → Add penalty*: a penalty added after a (partial) payment was registered. Across all fines with both activities, a payment before the penalty occurs in 16–45% of fines created in 2000–2006. For fines created from 2007 onwards, this falls to about 1% (Table III.6.c; it is also visible as DFR 8 *Payment → Add penalty* in Section 2). This points to a rule or system change around late 2006: once a payment is registered, no penalty is added afterwards.
    - __Variant 7__ (*Create Fine → Send Fine → Payment*) accounts for 3–11% of fines in 2000–2002 and practically disappears from 2003.
    - __Variant 2__ (immediate payment) grows from about 20% (2000–2002) to 32–38% (2004–2010). Variant 4 (penalty then payment) also gains share from 2004.
    - __Variant 1__ (credit collection) and __Variant 9__ fall to zero from 2012. This is the end bias, not a process change.
1. __Outcome__: as in BPIC17, every variant is either always or never positive, because the outcome is defined by the activities in the trace. This is a direct consequence of the outcome definition and not a finding about the process. These plots mainly show the overall outcome rate in the top panel.

In [ ]:
var_df = rtfm_plots.get_var_df(max_k=MAX_K, counts=True)
var_df['share'] = (var_df['variant count'] / log[CASE_ID].nunique()).round(3)
N_VARIANTS = log.groupby(CASE_ID)[ACT].apply(tuple).nunique()
print(f"{N_VARIANTS} distinct variants; top {MAX_K} cover {var_df['share'].sum():.1%} of cases")
save_table(var_df, 'table_1_top_variants')

#### 1.1 Representation type: __Throughput Time (TT)__

In [ ]:
fig('01_1_variants_tt')
rtfm_plots.topK_variants_evol(plt_type='type_tt', max_k=MAX_K, **common)

#### 1.2 Representation type: __Outcome__

##### 1.2.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('01_2_variants_outcome_A_paid')
rtfm_plots.topK_variants_evol(plt_type='type_outcome', max_k=MAX_K, **common)

##### 1.2.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('01_2_variants_outcome_B_credit_collection')
rtfm_plots.topK_variants_evol(plt_type='type_outcome', max_k=MAX_K, **common)

##### 1.2.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('01_2_variants_outcome_C_unresolved')
rtfm_plots.topK_variants_evol(plt_type='type_outcome', max_k=MAX_K, **common)

## __2. Directly-Follows Relations (DFRs)__
*[BPIC19 §2 | skipped in BPIC17 §0 because of lifecycle duplicates]*

BPIC17 skipped this log concept because its workflow activities are logged once per lifecycle transition, which makes self-loops like *W_X → W_X* dominate. RTFM has only `complete` events, so DFRs are meaningful here, as in BPIC19. The top 10 DFRs are printed below. As in BPIC19 §2, every representation type adds a left-hand panel with the fraction of cases containing the DFR and the mean number of occurrences per case. We also add the __outcome__ representation, which neither original notebook used.

The following trends, patterns and changes were identified:

1. __A fixed timer rule (DFR performance, 2.3).__ *Insert Fine Notification → Add penalty* (DFR 3) takes __exactly 60 days__ in every month of the log. The DFR performance plot shows a perfectly flat line: the penalty is added by a fixed rule 60 days after notification. A flat performance line is a strong signal of an automated, rule-driven step, and nothing similar is reported for BPIC17 or BPIC19.
1. __Gradual performance drifts (2.3).__
    - *Create Fine → Send Fine* (DFR 1), the delay before a fine is posted: about 55–65 days for fines created in 2000–2002, about 95–125 days in 2003–2006, 87–104 days in 2007–2009, then back down to 57–67 days in 2010–2012.
    - *Send Fine → Insert Fine Notification* (DFR 2) drops from about 25 days (2000–2001) to 12–17 days from 2002 (with 2011 as an exception at about 25 days).
    - *Add penalty → Payment* (DFR 6), late payment after the penalty: about 50 days until mid-2003, then a sudden jump to about 170–330 days from the fourth quarter of 2003. This is another change around the end of 2003 / start of 2004 (cf. Sections 3.3 and 5.3).
1. __Batch processing again (2.3).__ *Add penalty → Send for Credit Collection* (DFR 4) shows the same saw-tooth as the throughput time of credit-collection cases (Section 0): its performance is simply the waiting time until the next collection batch.
1. __Sudden control-flow drift (left-hand panels).__ *Payment → Add penalty* (DFR 8) and *Insert Fine Notification → Payment* (DFR 9) occur in 2–6% of fines created in 2000–2006 and practically vanish from 2007 (≤ 0.2%). These two DFRs together form the path *notification → partial payment → penalty* that disappears with variants 6 and 10 (Section 1, Table III.6.c). The DFR view localises the change precisely: a penalty is no longer added after a payment has been registered.
1. __Repeated activity (BPIC19 analogue).__ *Payment → Payment* (DFR 7) occurs in about 2–3% of fines in every year, about 1.07 times per affected fine. BPIC19 §2 found similar self-loops (*Record Service Entry Sheet*) and suggested data-entry artefacts.
1. __TT and NEPC (2.1, 2.2).__ Fines containing *Create Fine → Payment* (DFR 5) take about 12 days and have 2 events, against about 490 days for all others. Every DFR on the penalty path (DFRs 1–4) marks fines that are slower (490–690 days) and longer (4.5–5.2 events). These differences are stable over time, apart from the end-of-log collapse.
1. __Outcome (2.4).__ Mostly __process rules__, as with activities: DFR 4 always means credit collection, DFR 5 always means paid. DFR 6 (*Add penalty → Payment*) shows that a payment after the penalty almost always prevents collection (92% paid).

In [ ]:
dfr_df = rtfm_plots.get_DFR_df(max_k=MAX_K, counts=True)
save_table(dfr_df, 'table_2_top_dfrs')

#### 2.1 Representation type: __Throughput Time (TT)__

In [ ]:
fig('02_1_dfr_tt')
rtfm_plots.topK_dfr_evol(plt_type='type_tt', max_k=MAX_K, **common)

#### 2.2 Representation type: __Case Length (NEPC)__

In [ ]:
fig('02_2_dfr_nepc')
rtfm_plots.topK_dfr_evol(plt_type='type_events_case', max_k=MAX_K, **common)

#### 2.3 Representation type: __DFR Performance__

In [ ]:
fig('02_3_dfr_performance')
rtfm_plots.topK_dfr_evol(plt_type='type_dfr_performance', max_k=MAX_K, **common)

#### 2.4 Representation type: __Outcome__ (not used in either original notebook)

##### 2.4.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('02_4_dfr_outcome_A_paid')
rtfm_plots.topK_dfr_evol(plt_type='type_outcome', max_k=MAX_K, **common)

##### 2.4.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('02_4_dfr_outcome_B_credit_collection')
rtfm_plots.topK_dfr_evol(plt_type='type_outcome', max_k=MAX_K, **common)

##### 2.4.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('02_4_dfr_outcome_C_unresolved')
rtfm_plots.topK_dfr_evol(plt_type='type_outcome', max_k=MAX_K, **common)

## __3. Categorical Case Features__ 
*[BPIC17 §2 | BPIC19 §3]*
RTFM has 2 intrinsic categorical case features, `'case:vehicleClass'` and `'case:article'`. As in BPIC17, we added `'case:outcome'` (3 levels) to examine how the outcome relates to throughput time and case length over time.

### __3.1 Categorical Case Features: 'case:outcome'__
1. __Fraction of cases__: 44.9% Paid, 39.2% Credit Collection, 15.8% Unresolved. Unlike BPIC17, this distribution is __not stable__ over time (see the general trends in Section 0). Credit Collection falls from about 60% to about 30% between 2000 and 2005 and then recovers. Unresolved jumps from about 5% to about 20% during 2001–2002. Paid peaks at 50–55% in 2004–2006. From 2012, the shares collapse towards Paid and Unresolved because of the end bias.
1. __Mean Throughput Time (TT)__:
    - __Credit Collection__: by far the slowest (mean about 690 days), with the clear __saw-tooth pattern__ of the batches. Each tooth spans the interval between two collection batches. The tallest teeth (around 2006 and 2009) follow the years without a batch (2005 and 2008), when fines simply had to wait longer.
    - __Paid__: consistently fast (about 100 days), and stable over the entire log.
    - __Unresolved__: mostly 100–200 days. This is the time until the fine is sent; after that, nothing more is recorded.
1. __Mean Case Length (NEPC)__: Credit Collection cases have almost exactly __5 events__ in every month, since nearly all of them follow variant 1. Paid cases average about 3 events and Unresolved cases about 2.5. Unlike BPIC17, long duration and many events go together here: the slowest class is also the longest.

In [ ]:
level_summary('case:outcome')

#### __3.1.1 'case:outcome'__ - Representation type: __Throughput Time (TT)__

In [ ]:
fig('03_1_1_outcome_tt')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:outcome', plt_type='type_tt', max_k=MAX_K, **common)

#### __3.1.2 'case:outcome'__ - Representation type: __Case Length (NEPC)__

In [ ]:
fig('03_1_2_outcome_nepc')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:outcome', plt_type='type_events_case', max_k=MAX_K, **common)

### __3.2 Categorical Case Features: 'case:vehicleClass'__
`'case:vehicleClass'` has 4 levels: __A__ (97.6% of fines), __C__ (1.6%), __M__ (0.8%) and __R__ (4 fines in total). We plot the top 3 levels at a `'quarterly'` frequency, because at monthly resolution the minority classes contain only a handful of fines per month.

1. __Fraction of cases__: class A dominates throughout (92–99% per quarter). Class C drops from about 2–4% of fines (2000–2003) to about 1% from 2004, another change that coincides with the spring 2004 recoding (Table III.6.b).
1. __TT__ and __NEPC__: fines for classes C and M take longer on average (396 and 422 days, against 340 for A) and have more events (4.2–4.4 against 3.7). These differences are visible in most quarters but noisy.
1. __Outcome__: fines for classes C and M are paid less often (34–35% against 45%) and end in credit collection more often (50–52% against 39%). In the quarterly series this gap is clearest between 2004 and 2009. Before 2004 and from 2010, the lines cross regularly, so the gap is not stable over time.

Without a code list for the classes, we cannot say which vehicle types C and M stand for. What matters for the analysis is that the minority classes behave differently from the majority class.

In [ ]:
level_summary('case:vehicleClass')

#### __3.2.1 'case:vehicleClass'__ - Representation type: __Throughput Time (TT)__

In [ ]:
fig('03_2_vehicleclass_tt')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:vehicleClass', plt_type='type_tt', max_k=3, frequency='quarterly', time_unit=TIME_UNIT, case_assignment=CASE_ASSIGNMENT, numeric_agg=NUMERIC_AGG)

#### __3.2.2 'case:vehicleClass'__ - Representation type: __Case Length (NEPC)__

In [ ]:
fig('03_2_vehicleclass_nepc')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:vehicleClass', plt_type='type_events_case', max_k=3, frequency='quarterly', time_unit=TIME_UNIT, case_assignment=CASE_ASSIGNMENT, numeric_agg=NUMERIC_AGG)

#### __3.2.3 'case:vehicleClass'__ - Representation type: __Outcome__

##### 3.2.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('03_2_vehicleclass_outcome_A_paid')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:vehicleClass', plt_type='type_outcome', max_k=3, frequency='quarterly', time_unit=TIME_UNIT, case_assignment=CASE_ASSIGNMENT, numeric_agg=NUMERIC_AGG)

##### 3.2.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('03_2_vehicleclass_outcome_B_credit_collection')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:vehicleClass', plt_type='type_outcome', max_k=3, frequency='quarterly', time_unit=TIME_UNIT, case_assignment=CASE_ASSIGNMENT, numeric_agg=NUMERIC_AGG)

##### 3.2.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('03_2_vehicleclass_outcome_C_unresolved')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:vehicleClass', plt_type='type_outcome', max_k=3, frequency='quarterly', time_unit=TIME_UNIT, case_assignment=CASE_ASSIGNMENT, numeric_agg=NUMERIC_AGG)

### __3.3 Categorical Case Features: 'case:article'__
`'case:article'` is the article of the Italian Highway Code that was violated (66 levels). The four most frequent are Art. 157 (45.3%), Art. 7 (28.5%), Art. 158 (17.9%) and Art. 142 (3.8%). Under the Italian Highway Code, Arts. 7, 157 and 158 concern traffic rules in built-up areas and stopping and parking, and Art. 142 concerns speed limits. Check the official text before relying on these meanings.

1. __Fraction of cases: two sudden drifts.__
    - __May 2004__: Art. 157 jumps from about 10% to about 75% of new fines, while Art. 158 falls from about 40–50% to about 6%. The two articles seem to swap roles from one month to the next. That is far too abrupt for a change in driver behaviour and points to a change in how fines are coded. It coincides with the `lastSent` recoding in Section 5.3 (Table III.6.b).
    - __November 2009__: Art. 157 falls back from about 45% to 10–20%, and Art. 7 grows to about 50% from 2010. Art. 158 recovers to about 30%.
    - Smaller articles become much rarer after 2004: Art. 142 (speed) drops to near zero apart from a small bump in 2007–2008, and Arts. 171 and 172 almost disappear.
1. __TT__ and __NEPC__: the three large articles behave alike (316–353 days, 3.6–3.8 events). Several smaller articles are clearly slower and longer, e.g. Art. 142 (506 days, 4.8 events) and Arts. 146 and 172 (over 500 days).
1. __Outcome__: Art. 142 fines are paid much less often (23%) and go to credit collection much more often (60%) than the large parking-type articles (Paid about 42–48%, Credit Collection about 36–42%). Arts. 180 and 171 show similar patterns (Credit Collection 63–66%). The small-article series are noisy at monthly resolution; the summary table above is the more reliable view.

In [ ]:
level_summary('case:article')

#### __3.3.1 'case:article'__ - Representation type: __Throughput Time (TT)__

In [ ]:
fig('03_3_article_tt')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:article', plt_type='type_tt', max_k=MAX_K, **common)

#### __3.3.2 'case:article'__ - Representation type: __Case Length (NEPC)__

In [ ]:
fig('03_3_article_nepc')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:article', plt_type='type_events_case', max_k=MAX_K, **common)

#### __3.3.3 'case:article'__ - Representation type: __Outcome__

##### 3.3.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('03_3_article_outcome_A_paid')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:article', plt_type='type_outcome', max_k=MAX_K, **common)

##### 3.3.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('03_3_article_outcome_B_credit_collection')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:article', plt_type='type_outcome', max_k=MAX_K, **common)

##### 3.3.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('03_3_article_outcome_C_unresolved')
rtfm_plots.topK_categorical_caseftr_evol(case_feature='case:article', plt_type='type_outcome', max_k=MAX_K, **common)

## __4. Numerical Case Features__ 
*[BPIC17 §3 | not in BPIC19: no numerical case features]*
RTFM has 2 numerical case features: `'case:amount'`, the fine amount at creation in EUR (median 35, mean 44.7, max 4,351), and `'case:points'`, the penalty points deducted from the driving licence (0 for 97.6% of fines).

1. __Mean value over time__:
    - `'case:amount'` rises gradually. The modal amount for the common parking-type fines steps up every two years: 31.3 (2000), 32.8 (2001), 32.0 (2002), 33.6 (2003–04), 35 (2005–06), 36 (2007–08), 38 (2009–10), 39 (2011–12) and 41 EUR (2013). This fits the biennial inflation adjustment of fine amounts under Italian law; the 2002 dip coincides with the switch from lire to euro. The monthly mean also shows spikes from a few very large fines.
    - `'case:points'` is __exactly 0 until July 2003__. The first fine with points dates from 29 July 2003, which matches the introduction of the Italian penalty-point driving licence (*patente a punti*) on 1 July 2003. This is an external, regulatory concept drift captured by the log. The share of fines with points then grows to about 5–6% from 2008.
1. __TT__ and __NEPC__ ratios: as in BPIC17, these mostly combine the two trends: amounts rise slowly while TT has no clear trend. They add little beyond the separate series.
1. __Outcome__:
    - __Credit Collection__: credit-collection fines have a consistently higher mean amount (49.0 EUR against 40.1 for Paid) and more points. Fines with points go to credit collection 62% of the time, against 39% for fines without. Higher fines and fines that affect the licence are less often paid.
    - __Paid__: the mirror image, with lower amounts and fewer points.
    - __Unresolved__: close to Credit Collection (47.3 EUR, 0.11 points on average).

In [ ]:
save_table(log.drop_duplicates(CASE_ID)[numeric_casefeatures].describe().round(2), 'table_4_numeric_case_features')

### __4.1 'case:amount' and 'case:points'__ - Representation type: __Throughput Time (TT)__

In [ ]:
fig('04_1_numcase_tt')
rtfm_plots.num_casefts_evol(numeric_case_list=numeric_casefeatures, plt_type='type_tt', **common)

### __4.2 'case:amount' and 'case:points'__ - Representation type: __Case Length (NEPC)__

In [ ]:
fig('04_2_numcase_nepc')
rtfm_plots.num_casefts_evol(numeric_case_list=numeric_casefeatures, plt_type='type_events_case', **common)

### __4.3 'case:amount' and 'case:points'__ - Representation type: __Outcome__

##### 4.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('04_3_numcase_outcome_A_paid')
rtfm_plots.num_casefts_evol(numeric_case_list=numeric_casefeatures, plt_type='type_outcome', **common)

##### 4.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('04_3_numcase_outcome_B_credit_collection')
rtfm_plots.num_casefts_evol(numeric_case_list=numeric_casefeatures, plt_type='type_outcome', **common)

##### 4.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('04_3_numcase_outcome_C_unresolved')
rtfm_plots.num_casefts_evol(numeric_case_list=numeric_casefeatures, plt_type='type_outcome', **common)

## __5. Categorical Event Features__ 
*[BPIC17 §4 | BPIC19 §4]*
For event features, a case "has" a level if at least one of its events has it. The cell below shows on which activities each feature is recorded. This matters for interpretation: `org:resource` is only recorded when the fine is created, so it identifies the issuing officer rather than everyone who handled the case.

In [ ]:
for f in categorical_eventfeatures:
    nn = log[log[f].notna()]
    print(f"{f}: {nn[f].nunique()} levels, recorded on {nn[ACT].value_counts().to_dict()}")

### __5.1 Categorical Event Features: 'org:resource'__
`'org:resource'` has 147 distinct values in practice and is recorded on *Create Fine* (and on *Appeal to Judge*, where it is mostly a different code). The top 10 resources issue 40% of all fines.

1. __Fraction of cases per resource__:
    - __Arrivals and departures__: resources 559, 557, 561 and 558 only start issuing fines in 2004–2005. Resource 536 almost stops after 2005, and 558 issues no fines after 2008. This is the turnover of officers over 13 years.
    - __2010 gap, a sudden drift__: in 2010, none of the top 10 resources (except 49) issues a single fine. A different set of IDs (63, 858, 850, 851, 847, …) takes over for that year only. From 2011 the familiar IDs return. A complete replacement of the workforce for exactly one year is implausible, so the resource identifiers were most likely recoded in 2010. This is a data-quality issue to check before any resource-based analysis.
1. __TT__ and __NEPC__: for most resources, the mean TT and NEPC of their fines follow the overall line closely. The issuing officer has little effect on how long a fine takes.
1. __Outcome__: over the whole log, credit-collection rates per resource range from 28% (561) to 50% (536). The monthly plots show that this is __mostly an effect of *when* each resource was active__, not of the resource itself:
    - Resources 536 and 538 were most active in 2000–2003, when credit collection was at its peak (about 60%).
    - Resources 557, 558 and 561 were most active in 2004–2008, when it was lowest (about 30%).
    - Within each month, their lines mostly overlap with the line for all other resources.

    This is a good example of why the monthly comparison in DyLoPro is more informative than a single summary table: an overall difference can be explained entirely by the time period a level was active in.

In [ ]:
level_summary('org:resource')

#### __5.1.1 'org:resource'__ - Representation type: __Throughput Time (TT)__

In [ ]:
fig('05_1_resource_tt')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='org:resource', plt_type='type_tt', max_k=MAX_K, **common)

#### __5.1.2 'org:resource'__ - Representation type: __Case Length (NEPC)__

In [ ]:
fig('05_1_resource_nepc')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='org:resource', plt_type='type_events_case', max_k=MAX_K, **common)

#### __5.1.3 'org:resource'__ - Representation type: __Outcome__

##### 5.1.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('05_1_resource_outcome_A_paid')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='org:resource', plt_type='type_outcome', max_k=MAX_K, **common)

##### 5.1.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('05_1_resource_outcome_B_credit_collection')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='org:resource', plt_type='type_outcome', max_k=MAX_K, **common)

##### 5.1.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('05_1_resource_outcome_C_unresolved')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='org:resource', plt_type='type_outcome', max_k=MAX_K, **common)

### __5.2 Categorical Event Features: 'concept:name'__
RTFM has 11 activities, all plotted (top 10). *Create Fine* occurs in every case, so its "without" line is empty.

1. __Fraction of cases__:
    - *Send Fine* occurs in about 80% of fines created in 2000–2003, then in about 65% from 2004. This is the counterpart of the rise of immediate payments (variant 2) in 2004.
    - *Insert Fine Notification* and *Add penalty* always occur together (79,860 times each) and follow the same pattern.
    - The appeal activities (*Insert Date Appeal to Prefecture*, *Send Appeal to Prefecture*) grow slowly from about 1.6–1.8% of fines (2000–2001) to 2.6–4.1% (2006–2011).
    - *Send for Credit Collection* drops to zero from 2012 (end bias).
1. __TT__:
    - Fines __without__ *Send Fine* have a mean TT of only a few days: they were paid immediately.
    - Fines __with__ *Send for Credit Collection* show the saw-tooth pattern again.
    - Fines with an appeal are slower than other fines until about 2011.
1. __NEPC__: every activity except *Payment* adds steps, so fines containing it are longer. Fines with *Payment* are shorter, because most of them are immediate payments (2 events).
1. __Outcome__: most patterns here are __process rules__, not effects. Fines without *Send Fine* are almost never sent to credit collection, since a fine must be sent and notified first. *Payment* and *Send for Credit Collection* define the outcome classes themselves. *Add penalty* occurs in every credit-collection fine.

In [ ]:
level_summary('concept:name')

#### __5.2.1 'concept:name'__ - Representation type: __Throughput Time (TT)__

In [ ]:
fig('05_2_activity_tt')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='concept:name', plt_type='type_tt', max_k=MAX_K, **common)

#### __5.2.2 'concept:name'__ - Representation type: __Case Length (NEPC)__

In [ ]:
fig('05_2_activity_nepc')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='concept:name', plt_type='type_events_case', max_k=MAX_K, **common)

#### __5.2.3 'concept:name'__ - Representation type: __Outcome__

##### 5.2.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('05_2_activity_outcome_A_paid')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='concept:name', plt_type='type_outcome', max_k=MAX_K, **common)

##### 5.2.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('05_2_activity_outcome_B_credit_collection')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='concept:name', plt_type='type_outcome', max_k=MAX_K, **common)

##### 5.2.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('05_2_activity_outcome_C_unresolved')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='concept:name', plt_type='type_outcome', max_k=MAX_K, **common)

### __5.3 Categorical Event Features: 'lastSent'__
`'lastSent'` is recorded on *Insert Fine Notification* and has 3 codes: P, N and C. The log does not document what the codes mean.

1. __Fraction of cases: a sudden recoding.__ For fines created up to January 2004, the code is __always N__. For fines created from May 2004, it is __almost always P__ (95–99%) with a small share of C. The switch happens within February–April 2004. One code is replaced by another almost overnight, which indicates a change in coding or in the information system rather than in the process. It coincides with the Art. 157 / Art. 158 swap in Section 3.3 (Table III.6.b), so something changed in how fines were registered in spring 2004.
1. __TT, NEPC and outcome__: all fines with a `lastSent` value went through *Insert Fine Notification*, so they are the long fines (about 5.2 events, 560–640 days, 66–83% credit collection). Comparing P with N is mostly comparing two periods (before and after 2004), not two kinds of fines.

In [ ]:
level_summary('lastSent')

#### __5.3.1 'lastSent'__ - Representation type: __Throughput Time (TT)__

In [ ]:
fig('05_3_lastsent_tt')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='lastSent', plt_type='type_tt', max_k=MAX_K, **common)

#### __5.3.2 'lastSent'__ - Representation type: __Case Length (NEPC)__

In [ ]:
fig('05_3_lastsent_nepc')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='lastSent', plt_type='type_events_case', max_k=MAX_K, **common)

#### __5.3.3 'lastSent'__ - Representation type: __Outcome__

##### 5.3.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('05_3_lastsent_outcome_A_paid')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='lastSent', plt_type='type_outcome', max_k=MAX_K, **common)

##### 5.3.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('05_3_lastsent_outcome_B_credit_collection')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='lastSent', plt_type='type_outcome', max_k=MAX_K, **common)

##### 5.3.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('05_3_lastsent_outcome_C_unresolved')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='lastSent', plt_type='type_outcome', max_k=MAX_K, **common)

### __5.4 Categorical Event Features: 'dismissal'__
`'dismissal'` is recorded on *Create Fine*, *Send Appeal to Prefecture* and *Appeal to Judge*. `'NIL'` means the fine was not dismissed. Other codes are commonly read as a dismissal, e.g. `'#'` by the prefecture and `'G'` by a judge; other letters appear directly on *Create Fine*.

1. __Fraction of cases__: `'NIL'` occurs in practically every fine, because every *Create Fine* event carries it. The dismissal codes are rare (`'#'` in 1,980 fines, `'A'` 213, `'T'` 111, `'G'` 93, …).
1. __Outcome, a leakage pattern__: practically __every__ fine with a dismissal code other than `'NIL'` is 'Unresolved' (97–100% per code), with no payment and no credit collection. This is the RTFM equivalent of the `CreditScore` leakage in BPIC17: a dismissal code fully determines the outcome. Such a feature must not be used as an input to an outcome-prediction model unless its timing is controlled, i.e. only events before the prediction point are used.
1. __TT__: fines dismissed directly at creation (codes A, T, D) end after about 70–80 days, while fines dismissed after an appeal (`'#'`, `'G'`) take 200–280 days.

In [ ]:
level_summary('dismissal')

#### __5.4.1 'dismissal'__ - Representation type: __Throughput Time (TT)__

In [ ]:
fig('05_4_dismissal_tt')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='dismissal', plt_type='type_tt', max_k=MAX_K, **common)

#### __5.4.2 'dismissal'__ - Representation type: __Case Length (NEPC)__

In [ ]:
fig('05_4_dismissal_nepc')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='dismissal', plt_type='type_events_case', max_k=MAX_K, **common)

#### __5.4.3 'dismissal'__ - Representation type: __Outcome__

##### 5.4.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('05_4_dismissal_outcome_A_paid')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='dismissal', plt_type='type_outcome', max_k=MAX_K, **common)

##### 5.4.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('05_4_dismissal_outcome_B_credit_collection')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='dismissal', plt_type='type_outcome', max_k=MAX_K, **common)

##### 5.4.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('05_4_dismissal_outcome_C_unresolved')
rtfm_plots.topK_categorical_eventftr_evol(event_feature='dismissal', plt_type='type_outcome', max_k=MAX_K, **common)

## __6. Numerical Event Features__ 
*[BPIC17 §5 | BPIC19 §5]*
As in BPIC17, numerical event features can take several values per case, so each trace is collapsed to one value with `numEventFt_transform`. The cell below shows where each feature is recorded and how many values a case has:
- `'expense'`: the postal/administrative cost, recorded once, on *Send Fine*.
- `'amount'`: recorded on *Create Fine* (the original fine) and on *Add penalty* (the new total). The penalty __doubles__ the fine: the median ratio is 2.04.
- `'paymentAmount'`: recorded on every *Payment*; 1 to 6 payments per fine.
- `'totalPaymentAmount'`: running total of payments, recorded on *Create Fine* (0) and every *Payment*.

Choice of abstraction:
- `'expense'` and `'amount'`: __'max'__. For `expense` there is only one value per case. For `amount`, the max is the amount owed after any penalty.
- `'paymentAmount'`: __'sum'__, the total paid per fine (BPIC17-style choice of an aggregate).
- `'totalPaymentAmount'`: __'last'__, the final running total. This is exactly BPIC19's choice for its cumulative feature *Cumulative net worth (EUR)*. Since the running total also exists on *Create Fine*, fines without payments get 0 here instead of being left out.

__Important__: DyLoPro computes these series only over cases that have at least one value. Fines without *Send Fine* have no `expense`, and fines without payments have no `paymentAmount`; they are left out rather than counted as 0.

In [ ]:
for f in numeric_eventfeatures:
    nn = log[log[f].notna()]
    per_case = nn.groupby(CASE_ID).size()
    print(f"{f}: on {nn[ACT].value_counts().to_dict()} | cases with a value: {per_case.size:,} | values per case: {per_case.value_counts().sort_index().to_dict()}")

### __6.1 Numerical Event Features: 'expense' and 'amount' (max)__
1. __Mean value over time__:
    - `'expense'` rises __in clear steps__: about 8–9 EUR (2000–2001), about 11–12 (2002–2006), about 13 (2007–2010), about 16 (2011) and 18–20 (2012–2013). Steps this sharp point to tariff changes, e.g. postage or notification costs, rather than gradual drift.
    - `'amount'` (max, including penalty) rises slowly, like the case-level amount. It is also affected by how many fines received a penalty in each month.
1. __Outcome__:
    - `'amount'` is consistently and clearly higher for __credit-collection__ fines (mean 99 EUR against 52 for Paid fines). This is __mostly by construction__: every credit-collection fine received a penalty, which doubles the amount. As with `dismissal`, the feature carries information about the outcome path, so the value after *Add penalty* should not be used to predict credit collection.
    - `'expense'` shows no consistent difference between outcome classes. Both lines follow the same tariff steps.
1. __TT__ and __NEPC__ ratios mainly reflect the step changes in `expense` and add little else.

In [ ]:
subset_NumEvFtrs = ['expense', 'amount']

#### __6.1.1__ Representation type: __Throughput Time (TT)__

In [ ]:
fig('06_1_expense_amount_max_tt')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_tt', numEventFt_transform='max', **common)

#### __6.1.2__ Representation type: __Case Length (NEPC)__

In [ ]:
fig('06_1_expense_amount_max_nepc')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_events_case', numEventFt_transform='max', **common)

#### __6.1.3__ Representation type: __Outcome__

##### 6.1.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('06_1_expense_amount_max_outcome_A_paid')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_outcome', numEventFt_transform='max', **common)

##### 6.1.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('06_1_expense_amount_max_outcome_B_credit_collection')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_outcome', numEventFt_transform='max', **common)

##### 6.1.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('06_1_expense_amount_max_outcome_C_unresolved')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_outcome', numEventFt_transform='max', **common)

### __6.2 Numerical Event Features: 'paymentAmount' (sum)__
1. __Mean value over time__: the mean total paid per paying fine fluctuates around 45–60 EUR and rises slowly with the fine amounts. Spikes come from single months with a few very large payments.
1. __Outcome__:
    - __Paid__ vs. __not paid__: among fines that have payments, the "not paid" group consists only of the 2,126 credit-collection fines with a partial payment. Their totals are similar to those of paid fines. So the plot does __not__ show that non-paying fines pay less; those fines are simply absent from it. This shows the importance of the "cases without a value are dropped" rule above.
    - __Credit Collection__: same remark.
    - __Unresolved__: not plotted, because no unresolved fine has a payment by definition.

In [ ]:
subset_NumEvFtrs = ['paymentAmount']

#### __6.2.1__ Representation type: __Throughput Time (TT)__

In [ ]:
fig('06_2_payment_sum_tt')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_tt', numEventFt_transform='sum', **common)

#### __6.2.2__ Representation type: __Case Length (NEPC)__

In [ ]:
fig('06_2_payment_sum_nepc')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_events_case', numEventFt_transform='sum', **common)

#### __6.2.3__ Representation type: __Outcome__

##### 6.2.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('06_2_payment_sum_outcome_A_paid')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_outcome', numEventFt_transform='sum', **common)

##### 6.2.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('06_2_payment_sum_outcome_B_credit_collection')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_outcome', numEventFt_transform='sum', **common)

##### 6.2.3.C Outcome - __'case unresolved'__
Not plotted: by definition no unresolved fine has a payment, so there are no positive cases with a value to compare.

### __6.3 Numerical Event Feature: 'totalPaymentAmount' (last)__
*[mirrors BPIC19 §5: cumulative feature with `numEventFt_transform='last'`]*

`totalPaymentAmount` is a running total: it is 0 on *Create Fine* and is updated on every *Payment*, so its last value equals the sum of `paymentAmount` (99.9% of fines). Unlike Section 6.2, fines without payments are __included__ here, with value 0, because the running total also exists on *Create Fine*. This is the same reason BPIC19 chose `'last'` for *Cumulative net worth (EUR)*.

1. __Mean value over time__: the mean amount paid per fine (non-payers included) rises from about 18–20 EUR (2000–2002) to about 22–28 EUR from 2003, following the higher share of paid fines from 2004 (Section 3.1). For fines created in 2012 it jumps to about 43 EUR, and for 2013 it falls to about 17 EUR. Both are end-of-log effects: in 2012 most fines with a known ending are quick payers, and in 2013 most fines are still unresolved.
1. __Outcome__: credit-collection fines have a mean total payment of about 2 EUR (a few partial payments), against about 30–55 EUR per month for all other fines. The gap is consistent over the whole log. As with `paymentAmount`, this is __by construction__, since payment is part of the outcome definition, and not a usable predictor.
1. __Comparison with Section 6.2__: the same underlying data gives a very different picture depending on the abstraction. `paymentAmount` with `'sum'` drops non-payers and shows about 50 EUR with no outcome difference; `totalPaymentAmount` with `'last'` keeps them and shows a large outcome gap. The abstraction operation (paper Sect. 4.2) is therefore an analysis decision that must be reported, not a technical detail.

In [ ]:
subset_NumEvFtrs = ['totalPaymentAmount']

#### __6.3.1__ Representation type: __Throughput Time (TT)__

In [ ]:
fig('06_3_totalpayment_last_tt')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_tt', numEventFt_transform='last', **common)

#### __6.3.2__ Representation type: __Case Length (NEPC)__

In [ ]:
fig('06_3_totalpayment_last_nepc')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_events_case', numEventFt_transform='last', **common)

#### __6.3.3__ Representation type: __Outcome__

##### 6.3.3.A Outcome - __'case paid'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case paid')
fig('06_3_totalpayment_last_outcome_A_paid')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_outcome', numEventFt_transform='last', **common)

##### 6.3.3.B Outcome - __'case credit collection'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case credit collection')
fig('06_3_totalpayment_last_outcome_B_credit_collection')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_outcome', numEventFt_transform='last', **common)

##### 6.3.3.C Outcome - __'case unresolved'__

In [ ]:
rtfm_plots.set_outcomeColumn(outcome='case unresolved')
fig('06_3_totalpayment_last_outcome_C_unresolved')
rtfm_plots.num_eventfts_evol(numeric_event_list=subset_NumEvFtrs, plt_type='type_outcome', numEventFt_transform='last', **common)

## __7. One-year zoom at the originals' resolution (weekly) for 2009__
*[same resolution as BPIC17 and BPIC19: one year, `frequency='weekly'`]*

Sections 0–6 use monthly intervals because the log spans 13.5 years (deviation log, III.5). To compare with the originals at the same resolution, we rebuild the DyLoPro object on the fines created in __2009__ only and repeat four key plots weekly. 2009 has about 13,000 fines (about 250 per week, against about 600 in BPIC17). All of them had at least 3.4 years to finish before the log ends, and the year lies after the 2004 and 2006 changes and before the 2010 resource recoding.

In [ ]:
first_ts = log.groupby(CASE_ID)[TS].transform('min')
zoom_mask = first_ts.dt.year == ZOOM_YEAR
log_zoom = log[zoom_mask]
print(f"{log_zoom[CASE_ID].nunique():,} fines created in {ZOOM_YEAR}")
zoom_plots = dlp.DynamicLogPlots(log_zoom, case_id_key=CASE_ID, activity_key=ACT, timestamp_key=TS,
                                 categorical_casefeatures=categorical_casefeatures,
                                 numerical_casefeatures=numeric_casefeatures,
                                 categorical_eventfeatures=categorical_eventfeatures,
                                 numerical_eventfeatures=numeric_eventfeatures, outcome='case credit collection')
zoom_common = dict(frequency=ZOOM_FREQ, time_unit=TIME_UNIT, case_assignment=CASE_ASSIGNMENT, numeric_agg=NUMERIC_AGG)

#### 7.1 'case:outcome' - Throughput Time (weekly)

In [ ]:
fig('07_1_zoom_outcome_tt')
zoom_plots.topK_categorical_caseftr_evol(case_feature='case:outcome', plt_type='type_tt', max_k=MAX_K, **zoom_common)

#### 7.2 Variants - Throughput Time (weekly)

In [ ]:
fig('07_2_zoom_variants_tt')
zoom_plots.topK_variants_evol(plt_type='type_tt', max_k=MAX_K, **zoom_common)

#### 7.3 DFRs - DFR Performance (weekly)

In [ ]:
fig('07_3_zoom_dfr_performance')
zoom_plots.topK_dfr_evol(plt_type='type_dfr_performance', max_k=MAX_K, **zoom_common)

#### 7.4 'concept:name' - Outcome 'case credit collection' (weekly)

In [ ]:
fig('07_4_zoom_activity_outcome_cc')
zoom_plots.topK_categorical_eventftr_evol(event_feature='concept:name', plt_type='type_outcome', max_k=MAX_K, **zoom_common)

__Observations (Section 7)__:
1. __Seasonality inside one year (7.1, top panel)__: weekly volume rises from about 100–200 fines (January–February) to about 450–650 (June–August) and falls back to about 70–130 from October. Winter weeks are small samples, so their measures are noisier than in BPIC17.
1. __The exact collection cut-off becomes visible (7.1)__: the mean TT of credit-collection fines is about 600–680 days for fines created up to mid-March 2009, then __jumps to about 1,080 days__ and declines almost linearly to about 820 days by December. Our check confirms the mechanism: fines created in January–mid-March 2009 went in the October 2010 batch, and fines created from late March waited for the March 2012 batch. At monthly resolution (Section 3.1.1) this is one tooth of the saw-tooth; weekly, the batch cut-off can be dated to the week. The same step shows up in *Add penalty → Send for Credit Collection* (7.3).
1. __Fines are also posted in batches (7.3)__: *Create Fine → Send Fine* shows its own saw-tooth within 2009 (about 120 days falling to about 65, a jump to about 135 in July, falling to about 70). Sending is batched: in 2009, 62% of all *Send Fine* events fall on just 10 dates with about 570–590 fines each (72% in 2005). This pattern is invisible at monthly resolution and only appears at the originals' weekly resolution.
1. __The fixed 60-day penalty rule holds week by week__ (*Insert Fine Notification → Add penalty*, 7.3).
1. __Paid fines' TT steps up in April (7.1)__: from about 40–70 days (January–March) to about 140–150 days. The median stays at 5–7 days. The rise in the mean comes from a larger share of late payers (paths other than *Create Fine → Payment*: 11–15% in January–March, 23–25% from April), not from slower payment in general.
1. __Variants and activities (7.2, 7.4)__: the same rules as in Sections 1 and 5.2 hold. No new control-flow change appears within 2009, which is consistent with the full-log view.

__Comparison with the originals__: at the same weekly resolution, DyLoPro again exposes an operational mechanism (batch processing) and a fixed rule, comparable in kind to the BPIC19 change at 2018-09. Weekly intervals are needed to see posting batches; monthly intervals are needed to see the 2003–2007 changes. Neither resolution alone gives the full picture (paper Sect. 4.1, footnote 3).

## __8. Robustness check: fines created 2000–2010 only__
*[No equivalent in the originals; addresses the end-of-log bias from Section 0]*

Fines created from 2011 onwards had too little time to reach credit collection before the log was extracted (Section 0). Here we rebuild the DyLoPro object on fines created before 2011 and repeat two key plots. If the collapse at the end of the outcome and TT series disappears while the other patterns remain, the collapse is an extraction artefact and the remaining patterns are not.

In [ ]:
log_robust = log[first_ts < pd.Timestamp(ROBUSTNESS_CUTOFF, tz='UTC')]
robust_plots = dlp.DynamicLogPlots(log_robust, case_id_key=CASE_ID, activity_key=ACT, timestamp_key=TS,
                                   categorical_casefeatures=categorical_casefeatures,
                                   numerical_casefeatures=numeric_casefeatures,
                                   categorical_eventfeatures=categorical_eventfeatures,
                                   numerical_eventfeatures=numeric_eventfeatures, outcome='case credit collection')
fig('08_1_robust_outcome_tt')
robust_plots.topK_categorical_caseftr_evol(case_feature='case:outcome', plt_type='type_tt', max_k=MAX_K, **common)

In [ ]:
fig('08_2_robust_vehicleclass_outcome_cc')
robust_plots.topK_categorical_caseftr_evol(case_feature='case:vehicleClass', plt_type='type_outcome', max_k=3, frequency='quarterly', time_unit=TIME_UNIT, case_assignment=CASE_ASSIGNMENT, numeric_agg=NUMERIC_AGG)

# __Part III — Replication tables__
*Report sections 2, 3, 5 and 6. Every table is also written to `results/` as CSV.*

These tables are the core material for the report:

| Report section | Table |
|---|---|
| 2. Original setup / 3. Replication approach | III.1 coverage, III.2 setup (paper vs ours), III.3 log statistics |
| 5. Results and comparison | III.4 findings comparison (themes, as the slides ask for case-study papers) |
| 6(a). Deviation log | III.5 |
| Appendix | III.6, figures in `figures/` |
| References / reproducibility | III.7 manifest (`results/run_manifest.json`) |

### III.1 Coverage: log concept × representation type
Which combinations from paper Table 1 each notebook actually uses. `ISO` (isolated) is part of every plot.

In [ ]:
rt_cols = ['TT', 'NEPC', 'OUT', 'DFRP']
coverage = pd.DataFrame([
    ['Variants',                    'TT, OUT',       'TT',            'TT, OUT',             'NEPC, DFRP: n/a (Table 1)'],
    ['Directly-follows relations',  '- (skipped)',   'TT, NEPC, DFRP','TT, NEPC, DFRP, OUT', 'BPIC17 §0 skips DFRs because of lifecycle duplicates'],
    ['Categorical case features',   'TT, NEPC, OUT', 'TT, NEPC',      'TT, NEPC, OUT',       ''],
    ['Numerical case features',     'TT, NEPC, OUT', '- (none in log)','TT, NEPC, OUT',      ''],
    ['Categorical event features',  'TT, NEPC, OUT', 'TT, NEPC',      'TT, NEPC, OUT',       ''],
    ['Numerical event features',    'TT, NEPC, OUT (mean, max)', 'TT, NEPC (last)', 'TT, NEPC, OUT (max, sum, last)', ''],
    ['Outcome available?',          'yes, 3 binary', 'no',            'yes, 3 binary',       ''],
], columns=['Log concept (paper Table 1)', 'BPIC17 notebook', 'BPIC19 notebook', 'This notebook (RTFM)', 'Note']).set_index('Log concept (paper Table 1)')
save_table(coverage, 'III_1_coverage')

### III.2 Setup: as reported vs. this replication (report sections 2 and 3)

In [ ]:
setup = pd.DataFrame([
    ['Data', 'BPIC17 (31,413 cases, 1,198,366 events) and BPIC19 (251,470 cases, 1,587,925 events)', 'Sect. 5, fn. 8-9',
             f"RTFM, {DATA_DOI}, downloaded {DOWNLOAD_DATE}, SHA-256 {DATA_SHA256[:16]}..."],
    ['Preprocessing', 'BPIC17: 3 binary outcomes from last offer activity, invalid cases dropped; BPIC19: timestamps to UTC, 2 columns to object, start_date 01/01/2018',
             'notebooks §0 + preprocess_case_studies.py', 'Own preprocess_rtfm(): local-date timestamps, case features copied from Create Fine, 3 binary outcomes (I.5)'],
    ['Log discretization p', 'weekly (default); some plots 2-weekly', 'Sect. 4.1 (Def. 8, fn. 3); notebooks §0',
             f"{FREQ} for 13.5-year log; {ZOOM_FREQ} for {ZOOM_YEAR} zoom; quarterly for rare levels"],
    ['Case assignment', 'first event', 'Sect. 4.1 (Def. 9)', CASE_ASSIGNMENT],
    ['Log concepts / representation types', '6 concepts x 5 types', 'Sect. 4.2, Table 1', 'see III.1'],
    ['Aggregation Agg(X)', 'mean (default)', 'Sect. 4.2', NUMERIC_AGG],
    ['Dimensions K', 'top-10 most frequent', 'Sect. 4.2; notebooks (max_k=10)', f'top-{MAX_K} (top-3 for vehicleClass)'],
    ['Outcome', 'binary; 3-class split one-vs-rest (Teinemaa et al.)', 'Sect. 4.2 "Outcome", Sect. 5', 'Paid / Credit Collection / Unresolved, one-vs-rest'],
    ['Numerical event abstraction', 'BPIC17: mean, max; BPIC19: last', 'Sect. 4.2; notebooks §5', 'max, sum, last'],
    ['Outlier handling (axis only)', 'xtr_outlier_rem: BPIC17 mostly True, BPIC19 mostly False', 'notebooks', 'True (default)'],
    ['Software', 'DyLoPro beta (pre-release)', 'notebooks, NOTE cell', f"DyLoPro {ENV['DyLoPro']}, pandas {ENV['pandas']}, pm4py {ENV['pm4py']}, Python {ENV['python']}"],
    ['Hardware', 'not reported', '-', ENV['processor'] + ' / ' + ENV['platform']],
    ['Code', 'DyLoPro + DyLoPro_CaseStudies repositories', 'Sect. 5 fn. 7, Sect. 6', '; '.join(f'{k}: {v}' for k, v in ORIGINAL_CODE.items())],
    ['Runs / seeds', '1 (deterministic)', '-', '1 (deterministic, no seeds needed)'],
    ['Analysis method', 'visual inspection, findings annotated in notebook markdown', 'Sect. 5; notebooks', 'visual inspection + numeric check tables (level_summary, Section 0)'],
], columns=['Item', 'Paper / original notebooks', 'Where', 'This replication']).set_index('Item')
save_table(setup, 'III_2_setup')

### III.3 Log statistics side by side

In [ ]:
n_res = log.loc[log[ACT] == 'Create Fine', 'org:resource'].nunique()
interval_counts = cases.groupby(cases['start'].dt.to_period('M' if FREQ == 'monthly' else 'W')).size()
stats = pd.DataFrame([
    ['Cases', '31,413 (Sect. 5)', '251,470 (Sect. 5)', f"{len(cases):,}"],
    ['Events', '1,198,366 (Sect. 5)', '1,587,925 (Sect. 5)', f"{len(log):,}"],
    ['Activities', '26 (nb §4.2)', '42 (nb §4.2)', f"{log[ACT].nunique()}"],
    ['Distinct variants', '> 15,000 (nb §1)', 'not reported', f"{N_VARIANTS:,}"],
    ['Share of most frequent variant', '3.4% (nb §1)', '25-30% weekly (Sect. 5)', f"{var_df['share'].iloc[0]:.1%}"],
    ['Mean events per case (NEPC)', '38.1 (derived)', '6.3 (Sect. 5)', f"{cases['n_events'].mean():.2f}"],
    ['Mean throughput time', 'not reported', '~80 days before 2018-09 (Sect. 5)', f"{cases['TT_days'].mean():.0f} days"],
    ['Resources', '148 (nb §4.1)', '628 (nb §4.1)', f"{n_res} (issuing officers)"],
    ['Time span', '2016 (1 year)', '2018 (1 year)', f"{cases['start'].min():%Y-%m} to {log[TS].max():%Y-%m}"],
    ['Intervals at chosen p', '~52 weekly', '~52 weekly', f"{len(interval_counts)} {FREQ}"],
    ['Cases per interval (median)', '~600 (derived)', '~4,800 (derived)', f"{interval_counts.median():.0f}"],
    ['Outcome split', 'Canceled 47% / Accepted 41% / Refused 12% (nb §2.1)', 'no outcome',
     ' / '.join(f'{k} {v:.0%}' for k, v in cases['outcome'].value_counts(normalize=True).items())],
], columns=['Statistic', 'BPIC17', 'BPIC19', 'RTFM (this notebook)']).set_index('Statistic')
save_table(stats, 'III_3_log_statistics')

*BPIC values are taken from the paper (Sect. 5) or the original notebooks (nb §). Values marked "derived" are computed from reported numbers (e.g. events ÷ cases).*

### III.4 Comparison of findings (themes), report section 5
For a case-study paper the comparison is by type of finding, not by number. *Verdict* states whether DyLoPro surfaces the same kind of finding on RTFM.

In [ ]:
findings = pd.DataFrame([
    ['Extraction bias at the end of the log (performance)',
     'BPIC19: weekly mean TT ~80 days, declines from 2018-09 to 0 (paper Sect. 5, Fig. 3; nb §0). BPIC17: TT declines near the end (nb §0)',
     f"Mean TT {per_year.loc[2011, 'mean TT (days)']:.0f} days (2011) -> {per_year.loc[2012, 'mean TT (days)']:.0f} (2012) -> {per_year.loc[2013, 'mean TT (days)']:.0f} (2013) (Sect. 0, Table 0.1); disappears when restricted to 2000-2010 (Sect. 8)",
     'Confirmed'],
    ['Extraction bias at the end of the log (outcome)',
     'Not reported (BPIC17 outcome described as stable, nb §2.1; BPIC19 has no outcome)',
     f"Credit-collection share {per_year.loc[2011, 'Credit Collection']:.0%} (2011) -> {per_year.loc[2012, 'Credit Collection']:.0%} (2012) -> {per_year.loc[2013, 'Credit Collection']:.0%} (2013): collection needs >= 273 days (Sect. 0.2)",
     'New finding (beyond paper)'],
    ['Sudden control-flow drift',
     'BPIC19: variant 1 drops, variant 6 rises around 2018-09 because a new start activity appears (paper Sect. 5, Fig. 3; nb §1, §4.2)',
     'Variants 6 and 10 and DFR Payment -> Add penalty disappear end 2006 (payment before penalty 16-45% -> ~1%, Table III.6.c); variant 7 disappears in 2003 (Sect. 1, 2)',
     'Confirmed (same type, different cause)'],
    ['Data leakage visible through the outcome representation',
     "BPIC17: 'CreditScore' is 0 for 3,719 of 3,720 refused cases (paper Sect. 5, Fig. 2; nb §5.2)",
     "Any non-NIL 'dismissal' code -> Unresolved in 97-100% of cases (Sect. 5.4); 'amount' (max) doubled by the penalty for credit-collection fines (Sect. 6.1)",
     'Confirmed'],
    ['Data perspective related to performance, stable until a break point',
     "BPIC19: 'Packaging' slower and 'Sales' faster than other spend areas, both break down at 2018-09 (paper Sect. 5, Fig. 4; nb §3)",
     'Art. 142 (speed) fines slower (506 vs ~330 days) and more often collected (60% vs ~36-42%); vehicle classes C/M slower; no shared break point (Sect. 3.2, 3.3)',
     'Partially confirmed (patterns found, no common break point)'],
    ['Resource-perspective drift',
     "BPIC17: sudden shifts in resource involvement, a new resource with a learning effect (nb §4.1). BPIC19: nothing notable (nb §4.1)",
     'Officer turnover 2004-2008; all top-10 resource IDs replaced for 2010 only, most likely a recoding (Sect. 5.1)',
     'Confirmed (drift); learning effect not examined'],
    ['Repeated activities in directly-follows relations',
     "BPIC19: DFRs 'Record Service Entry Sheet' and 'Record Goods Receipt' repeated many times per case, suggesting data-entry artefacts (nb §2)",
     'DFR Payment -> Payment (instalments) found (Sect. 2); domain knowledge says these are real instalments, not errors',
     'Pattern detected; interpretation differs'],
    ['Lifecycle duplication inflating variants and NEPC',
     'BPIC17: W_ activities logged per state change, > 15,000 variants (nb §0, §1)',
     f"Not applicable: all events 'complete'; only {N_VARIANTS} variants",
     'Not applicable'],
    ['Data-recording / coding changes',
     'Not reported',
     "Spring 2004: 'lastSent' N -> P and Art. 157/158 swap within weeks (Table III.6.b); 2010 resource recoding (Sect. 5.1)",
     'New finding (beyond paper)'],
    ['External (regulatory) and recurring drifts',
     'Not reported (paper Sect. 1 names sudden, gradual, recurring and incremental drift as targets)',
     "Points appear from July 2003 (penalty-point licence); fine amounts step up every two years; 36% of fines in July-August; credit-collection batches create a saw-tooth in TT (Sect. 0, 4)",
     'New finding (beyond paper)'],
    ['Operational mechanisms visible in DFR performance',
     'Not reported (BPIC19 nb §2 uses DFR performance only to compare DFRs)',
     'Insert Fine Notification -> Add penalty is exactly 60 days in every month (fixed rule); fines are posted in batches (72% of 2005 Send Fine events on 10 dates) and sent to collection in yearly batches (saw-tooth) (Sect. 2, 7)',
     'New finding (beyond paper)'],
], columns=['Type of finding', 'Original (paper / notebooks)', 'RTFM (this notebook)', 'Verdict']).set_index('Type of finding')
save_table(findings, 'III_4_findings_comparison')

### III.5 Deviation log, report section 6(a)

In [ ]:
deviations = pd.DataFrame([
    ['Case studies on BPIC17 and BPIC19 (Sect. 5)', 'RTFM log', 'Assignment: replicate on a different log; RTFM is the authors\' own announced but unpublished case study (README "COMING SOON")',
     'Tests generalisability (external validity); no number-to-number comparison with the paper possible'],
    ['Weekly intervals (default; nb §0); p must give populated sublogs (Sect. 4.1, fn. 3)', f"{FREQ} for the full log; {ZOOM_FREQ} for {ZOOM_YEAR} only (Sect. 7)",
     '13.5-year log would give ~700 weekly points', 'Effects shorter than a month are hidden in Sections 0-6; Section 7 shows them for one year'],
    ["Authors' preprocess_RTFM(): timestamps used as stored, in UTC", 'Converted to the Italian calendar date', 'Dates are stored as local midnight (22:00/23:00 UTC)',
     f"Authors' version puts {MONTH_SHIFT:.1%} of fines in the wrong month and {WEEK_SHIFT:.1%} in the wrong week"],
    ["Authors' preprocess_RTFM(): 'article' and 'points' cast to object (categorical), left on the Create Fine event",
     "Copied to all events as case features: 'case:article' categorical, 'case:points' numerical", 'Both describe the fine and are recorded once; points is a count',
     "Points plotted as a mean instead of per-level fractions"],
    ["Authors' preprocess_RTFM(): no outcome defined. BPIC17: outcome from last offer activity, invalid cases dropped (preprocess_bpic17)",
     '3 classes (Credit Collection / Paid / Unresolved) from activity presence, no cases dropped', 'RTFM has no outcome attribute; credit collection is the label used in the literature (Teinemaa et al.)',
     "'Unresolved' mixes still-running and dismissed fines; affected by the end-of-log bias"],
    ["Authors' preprocess_RTFM() keeps 'notificationType', 'totalPaymentAmount'", "'notificationType' not analysed; 'totalPaymentAmount' used with 'last'", 'notificationType is 99.8% one value',
     'None'],
    ['BPIC17 skips DFRs (lifecycle duplicates, nb §0)', 'DFR section included (Sect. 2)', 'All RTFM events are "complete"', 'More control-flow coverage than BPIC17'],
    ['DFR outcome representation: in Table 1, used in neither notebook', 'Used (Sect. 2.4)', 'Outcome available; completes the coverage', 'None'],
    ['distinct_variants_evol (library)', 'distinct_variants_AdvancedEvol (Sect. 0.4)', 'Deprecated from 0.1.1 (HISTORY.md)', 'None'],
    ['select_time_range() / start_date for sub-periods (BPIC19 uses start_date)', 'Own filter on the first-event date (Sect. 7, 8)', 'select_time_range() rejects valid ranges in 0.1.1; start_date/end_date keep only cases that also END within the range',
     'Our subsets keep long-running cases that end after the cut-off'],
    ['xtr_outlier_rem mostly False in BPIC19, mostly True in BPIC17', 'True (default)', 'Only rescales the y-axis', 'None on the data; occasional extreme points are clipped from view'],
    ['DyLoPro beta version, other versions not reported', f"DyLoPro {ENV['DyLoPro']}, pandas {ENV['pandas']}, Python {ENV['python']}", 'Beta not available; pandas 3 breaks DyLoPro 0.1.1',
     'None expected; all plot types ran'],
    ['Findings from visual inspection only', 'Visual inspection plus numeric check tables (level_summary, Section 0)', 'Reduce subjectivity of reading plots', 'Higher reliability of our claims'],
    ['Single run, no seeds reported', 'Single run, no seeds', 'DyLoPro is deterministic', 'None'],
], columns=['Paper / authors say', 'We did', 'Why', 'Expected impact'])
deviations.index = range(1, len(deviations) + 1)
save_table(deviations, 'III_5_deviation_log')

### III.6 Supporting tables

#### III.6.a Throughput time distribution per year of fine creation (end-of-log bias, Section 0)

In [ ]:
save_table(cases.groupby('year')['TT_days'].describe(percentiles=[.25, .5, .75, .95]).round(0), 'III_6a_tt_per_year')

#### III.6.b The spring-2004 recoding (Sections 3.3 and 5.3)

In [ ]:
create = log[log[ACT] == 'Create Fine'].assign(month=lambda d: d[TS].dt.to_period('M'))
window = create[(create['month'] >= '2003-10') & (create['month'] <= '2004-07')]
art = pd.crosstab(window['month'], window['case:article'], normalize='index')[['Art. 157', 'Art. 158', 'Art. 7']]
ls = log[log['lastSent'].notna()].merge(cases[['start']], left_on=CASE_ID, right_index=True)
ls = ls[(ls['start'].dt.to_period('M') >= '2003-10') & (ls['start'].dt.to_period('M') <= '2004-07')]
last_sent = pd.crosstab(ls['start'].dt.to_period('M'), ls['lastSent'], normalize='index').add_prefix('lastSent=')
save_table(art.join(last_sent).round(2), 'III_6b_2004_recoding')

#### III.6.c The late-2006 control-flow change (Sections 1 and 2)

In [ ]:
both = log[log[ACT].isin(['Payment', 'Add penalty'])].groupby(CASE_ID)[ACT].agg(list)
both = both[both.map(lambda l: 'Payment' in l and 'Add penalty' in l)]
pay_first = both.map(lambda l: l.index('Payment') < l.index('Add penalty'))
save_table(pay_first.groupby(cases.loc[pay_first.index, 'year']).mean().round(3).to_frame('payment before penalty').T, 'III_6c_payment_before_penalty')

### III.7 Export manifest
Everything needed to cite this run in the report's reproducibility paragraph.

In [ ]:
manifest = {**ENV, 'data file': str(DATA_PATH), 'data DOI': DATA_DOI, 'download date': DOWNLOAD_DATE,
            'data SHA-256': DATA_SHA256, 'original code': ORIGINAL_CODE,
            'parameters': {'frequency': FREQ, 'time_unit': TIME_UNIT, 'case_assignment': CASE_ASSIGNMENT,
                           'numeric_agg': NUMERIC_AGG, 'max_k': MAX_K, 'zoom_year': ZOOM_YEAR, 'zoom_frequency': ZOOM_FREQ,
                           'robustness_cutoff': ROBUSTNESS_CUTOFF}}
(RES_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print(f"\n{len(list(FIG_DIR.glob('*.png')))} figures in {FIG_DIR}/, {len(list(RES_DIR.glob('*.csv')))} tables in {RES_DIR}/")